# Residential minimum lot area

A parcel map with an SDK mobile composition and a computed headline statistic.

This is an editable worked example of [`minimum_lot_area.py`](../src/graphics/minimum_lot_area.py). It runs Python SDK calls directly—no website, CLI subprocess, or registered build dispatcher. The canonical definition remains the publishing source of truth.

**What the numbers mean:** The existing classifier evaluates recorded R1–R6 parcel areas, excludes the same parks and missing-data cases, and applies the existing 1% tolerance. It is not a parcel-specific legal determination.

Start with **Run All**, inspect the data table and preview, then change the title or a visual encoding in step 4. Parcel examples load the full city and can take several minutes.


## 1. Open the libraries

Use the environment in [README.md](README.md). Paths below locate the checkout, not a personal machine.


In [ ]:
from pathlib import Path
import sys

# Run from this notebook folder or anywhere inside the Detroit checkout.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "strongtowns-data.lock.json").is_file()
             and (p / "projects/graphics/src/graphics").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter inside the strongtowns-detroit checkout; see README.md.")
sys.path.insert(0, str(ROOT / "src"))
GRAPHICS = ROOT / "projects/graphics"
FORUM = ROOT / "projects/detroit-land-use-forum"
import strongtowns_graphics as graphics_sdk
if not hasattr(graphics_sdk, "GraphicInput"):
    raise RuntimeError(
        "This kernel has an older graphics SDK. Restart Jupyter with the uv command "
        "in README.md so it uses this project's locked dependencies."
    )
from IPython.display import SVG, display
from strongtowns_graphics import (
    GraphicFormat, GRAPHIC_FORMAT_SPECS, render_graphic_canvas,
    render_graphic_svg, write_graphic_bundle,
)

NOTEBOOK_NAME = 'minimum_lot_area'


In [ ]:
import sys

from pathlib import Path

import geopandas as gpd

import pandas as pd

PARCEL_DIR = FORUM / "parcel-geometry"

sys.path.insert(0, str(PARCEL_DIR))

from build_minimum_lot_size_asset import (  # noqa: E402
    build_graphic, classify_lot_area, select_residential_area_cases,
)

from strongtowns_graphics import (
    GraphicInput,
    MobileMapInset,
    MobileMapPocket,
    graphic_definition,
    map_on_mobile,
)


## 2. Resolve the prepared data

The aliases below name the inputs you will read. The data SDK resolves only the snapshots pinned by this project. Change prepared inputs through a reviewed data lock update, not by pointing at a moving latest file.


In [ ]:
from strongtowns_data import DataBuildSystem, DataLock, DataRepository
from strongtowns_detroit.repositories import data_repository
from strongtowns_graphics import GraphicBuildContext

requirements = (
        GraphicInput("parcels", "detroit.parcels", "accepted.parquet"),
        GraphicInput("histories", "detroit.bza.gemini.raw", "raw/case_histories.csv"),
        GraphicInput("categories", "detroit.bza.gemini.raw", "raw/case_categories.csv"),
    )
lock = DataLock.load(ROOT / "strongtowns-data.lock.json")
repository = DataRepository(DataBuildSystem.find(data_repository()))
paths, provenance = {}, {}
for requirement in requirements:
    try:
        reference = lock.asset(requirement.dataset_id)
        paths[requirement.alias] = repository.artifact(reference, requirement.artifact)
        if not paths[requirement.alias].is_file():
            raise FileNotFoundError(paths[requirement.alias])
        provenance[requirement.alias] = {
            "dataset": requirement.dataset_id,
            "artifact": requirement.artifact,
            "snapshot": reference.snapshot_id,
            "manifest_sha256": reference.manifest_sha256,
        }
    except (ValueError, KeyError, FileNotFoundError) as error:
        raise RuntimeError(
            f"Prepared input unavailable: {requirement.dataset_id}/{requirement.artifact}. "
            "Ask the data maintainer to restore the pinned snapshot in strongtowns-data; "
            "this notebook never fetches data or changes the lock."
        ) from error
context = GraphicBuildContext(paths)
provenance


## 3. Prepare and inspect the table

This follows the existing graphic’s data selection and calculations. The displayed rows are a preview; the graphic uses the full prepared table.


In [ ]:
frame = classify_lot_area(gpd.read_parquet(
    context.input("parcels"),
    columns=["parcel_id", "zoning_district", "total_square_footage",
             "taxpayer_1", "taxpayer_2", "geometry"],
))

histories = pd.read_csv(context.input("histories"))

categories = pd.read_csv(context.input("categories"))

cases = select_residential_area_cases(histories, categories)

evaluated = frame[frame["evaluated"]]

affected = int(evaluated["below_minimum"].sum())

total = len(evaluated)

affected_share = affected / total


In [ ]:
display(frame.drop(columns="geometry").head(8))


## 4. Build the graphic with the SDK

Edit `title`, `subtitle`, colors, legends, or explicit encodings here. Keep sources and descriptions accurate when changing data. The parcel and travel examples reuse existing project map-drawing helpers, then call SDK composition functions; the bar, point-map, and continuous-choropleth examples expose their renderer calls directly.


In [ ]:
base = build_graphic(
    frame,
    cases,
    title="Detroit's Zoning Code mandates suburban lot sizes",
    subtitle=(
        "Recorded R1–R6 parcel area compared with the 5,000-square-foot minimum"
    ),
    sources=(
        "Sources: City of Detroit parcel data; Detroit BZA minutes, "
        "2019–2026; Detroit Code §§50-13-1–7, 50-13-21.",
    ),
    description=(
        f"{affected:,} of {total:,} evaluated R1 through R6 parcels are "
        "more than one percent below 5,000 square feet. "
        f"{len(cases)} BZA case histories sought residential "
        "minimum-lot-area relief."
    ),
)

graphic = map_on_mobile(
    base,
    insets=(
        MobileMapInset.hero_statistic(
            pocket=MobileMapPocket.LOWER_RIGHT,
            value=f"{affected_share:.0%}",
            label=(
                "OF EVALUATED R1–R6 PARCELS",
                "FALL BELOW DETROIT'S",
                "MINIMUM LOT AREA",
            ),
        ),
    ),
)

graphics = {"minimum-lot-area": graphic}


## 5. Choose the Instagram format and preview

Feed uses 1080 × 1350; story uses 1080 × 1920 with the library’s established content padding. Changing this enum preserves the publishing policy.


In [ ]:
# Change to GraphicFormat.INSTAGRAM_STORY for a story-sized export.
TARGET = GraphicFormat.INSTAGRAM_POST
EXPORT_PNG = True  # SVG and HTML work without the rsvg-convert system tool.
target = GRAPHIC_FORMAT_SPECS[TARGET]


In [ ]:
# Preview exactly the composition used by the export below.
for name, graphic in graphics.items():
    print(name)
    svg = (render_graphic_canvas(
        graphic, canvas_aspect_ratio=target.aspect_ratio,
        content_aspect_ratio=target.content_aspect_ratio,
        content_top_padding=target.content_top_padding,
    ) if target.content_aspect_ratio else render_graphic_svg(
        graphic, aspect_ratio=target.aspect_ratio,
    ))
    display(SVG(svg))
    print("Alt text:", graphic.description or graphic.title_text)


## 6. Export

PNG is ready for Instagram; SVG and HTML retain the composition for inspection. Copy the adjacent alt-text file when posting. Exports go to the ignored `projects/graphics/output/notebooks/` directory and can be regenerated. Inspect all pages before sharing.


In [ ]:
import json
import shutil

output_dir = GRAPHICS / "output/notebooks" / NOTEBOOK_NAME / TARGET.value
formats = ("html", "svg", "png") if EXPORT_PNG else ("html", "svg")
if EXPORT_PNG and shutil.which("rsvg-convert") is None:
    raise RuntimeError(
        "PNG export needs rsvg-convert (see README.md). "
        "Set EXPORT_PNG = False above and rerun the export to save SVG/HTML now."
    )
for name, graphic in graphics.items():
    files = write_graphic_bundle(
        output_dir, name, graphic,
        aspect_ratio=target.aspect_ratio, png_width=target.png_width,
        content_aspect_ratio=target.content_aspect_ratio,
        content_top_padding=target.content_top_padding, formats=formats,
    )
    (output_dir / f"{name}.alt.txt").write_text(
        graphic.description or graphic.title_text, encoding="utf-8"
    )
    for kind, path in files.items():
        print(f"{kind}: {path}")
# Keep the exact source identities alongside your exported graphics.
(output_dir / "sources.json").write_text(
    json.dumps(provenance, indent=2) + "\n", encoding="utf-8"
)


## Try the pattern on another question

Make a copy of this notebook. Start by changing editorial wording, then inspect the explicit input table before changing a field or grouping. Keep units, unknown records, source coverage, and denominators visible. Changing geographic scope or a legal threshold requires reviewing the method and claim, not just replacing the title.

Use the other notebooks to compare stacked bars, categorized points, continuous parcel maps, and mobile map compositions. Clear outputs before committing a notebook; put publishing changes back into the canonical definition.
